
## -------------- Complete Research Workflow  ----------------

Evaluating the Dempster-Shafer  belief-based intent clarification system across multiple datasets (e.g., `banking77`, `clinc150`, `snips`, `atis`).

Select the dataset in the **Setup and Configuration** section; all steps, paths, and sample counts update accordingly.

### Phase 1: Configure Optimal DS Agent (One-Time Setup)

**STEP 0**: Vanilla Baseline (Pure LogisticRegression + E5-BERT)
- Train classifier on the selected dataset
- Evaluate without hierarchy or DS reasoning
- Establish baseline performance metrics

**STEP 1**: Extract Belief Values
- Initialize DS system with hierarchy
- Compute beliefs for all intents using DEFAULT thresholds
- Extract per-intent belief distributions for optimization
- Output: `{dataset}_beliefs.csv` (used for STEP 2)

**STEP 2**: Compute Optimal Thresholds
- Analyze belief distributions from STEP 1
- Find optimal threshold per intent using ancestor-aware ground truth
- Test multiple threshold values (0.0-1.0) to maximize F1
- Output: `{dataset}_optimal_thresholds.json` (used for STEP 3)

### Phase 2: Deploy DS Agent with Optimal Configuration

**STEP 3**: DS Evaluation WITH Optimal Thresholds
- Initialize DS system with optimal thresholds from STEP 2
- LLM-based user simulation for clarifications
- Measure: Accuracy, clarification rate, belief progression
- Output: `{dataset}_predictions.csv` (used for STEP 5)

**STEP 5**: Select User Study Queries
- Analyze STEP 3 results to identify challenging queries
- Prioritize by: low confidence, high interactions, incorrect predictions
- Select top N queries for real human validation
- Output: `selected_queries_for_user_study.csv` (used for STEP 6)

**STEP 6**: Real User Study (Streamlit Interactive Interface)
- Launch Streamlit app with selected challenging queries
- Real human participants replace LLM agent
- Track belief progression and collect human responses
- Validate predictions with actual user behavior

### Phase 3: Analysis & Reporting

**STEP 7**: Comparison Report
- Compare: Vanilla baseline vs DS with optimal thresholds vs Real users
- Analyze improvements in accuracy, clarification effectiveness
- Generate visualizations and summary metrics

---

## Key Differences from Baseline

| Aspect | Vanilla Baseline | DS with Optimal Thresholds | Real User Study |
|--------|------------------|---------------------------|-----------------|
| **What** | LogisticRegression + BERT | DS hierarchy + clarifications | Real humans replace LLM |
| **Queries** | All test queries (dataset-dependent) | Same test set | Selected challenging queries |
| **Clarifications** | None | LLM-simulated | Real human users |
| **Focus** | Baseline metrics | Optimized performance | Validation with humans |

---

**Demo Mode**: This notebook uses small sample sizes for quick testing. Set `QUICK_DEMO = False` for full evaluation.

## Setup and Configuration

Initialize paths, load dependencies, and configure the workflow.

In [2]:
from ipywidgets import Dropdown

dataset_selector = Dropdown(
    options=["banking77", "clinc150", "snips", "atis"],
    value="banking77",
    description="Dataset:",
)

dataset_selector

Dropdown(description='Dataset:', options=('banking77', 'clinc150', 'snips', 'atis'), value='banking77')

In [3]:
import sys
from pathlib import Path
import os

# Add project root to path
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

# Load .env early so all modules (including Dropbox) see credentials
# regardless of working directory
try:
    from dotenv import load_dotenv
    load_dotenv(project_root / ".env", override=False)
except ImportError:
    pass

import logging
from datasets import load_dataset
import pandas as pd
import json

from src.data.data_loader import DataLoader

# Configure logging (WARNING level to reduce output noise)
logging.basicConfig(level=logging.WARNING, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Silence noisy third-party loggers
logging.getLogger('sentence_transformers').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)
logging.getLogger('src.agents.customer_agent').setLevel(logging.WARNING)
logging.getLogger('src.models.ds_mass_function').setLevel(logging.WARNING)
logging.getLogger('config.threshold_loader').setLevel(logging.WARNING)

# Configuration
QUICK_DEMO = False  # Set to False for full evaluation

DATASET = dataset_selector.value if "dataset_selector" in globals() else "banking77"

# NOTE: DataLoader handles CLINC150 OOS filtering (intent != "oos"), matching SVM_DS_clinc150.
BASE_DIR = project_root
EXPERIMENTS_DIR = BASE_DIR / "experiments" / DATASET
RESULTS_DIR = BASE_DIR / "results" / DATASET / "workflow_demo"
CONFIG_DIR = BASE_DIR / "config"
USER_STUDY_DIR = BASE_DIR / "outputs" / "user_study" / "workflow_demo"

# Sample sizes
if QUICK_DEMO:
    TRAIN_SAMPLES = None  # Use full training set even in demo
    EVAL_SAMPLES = 50     # Quick evaluation
    LLM_SAMPLES = 100     # Quick LLM simulation
    HUMAN_STUDY_SAMPLES = 30  # Queries for human study
else:
    TRAIN_SAMPLES = None
    EVAL_SAMPLES = None   # Full test set
    LLM_SAMPLES = None    # Full test set
    HUMAN_STUDY_SAMPLES = 100

# Create directories
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
USER_STUDY_DIR.mkdir(parents=True, exist_ok=True)

# Dynamic dataset size for printing
data_loader = DataLoader(DATASET)
data_loader.load()
eval_total = len(data_loader.get_split_data("test")[0])

print(f"   Configuration complete")
print(f"   Mode: {'QUICK DEMO' if QUICK_DEMO else 'FULL EVALUATION'}")
print(f"   Dataset: {DATASET}")
print(f"   Evaluation samples: {EVAL_SAMPLES if EVAL_SAMPLES else f'All ({eval_total})'}")
print(f"   LLM simulation samples: {LLM_SAMPLES if LLM_SAMPLES else f'All ({eval_total})'}")
print(f"   Human study queries: {HUMAN_STUDY_SAMPLES}")


   Configuration complete
   Mode: FULL EVALUATION
   Dataset: clinc150
   Evaluation samples: All (4500)
   LLM simulation samples: All (4500)
   Human study queries: 100


====================================================================================================================================================================================================================

## STEP 0: Vanilla Baseline (Pure LogisticRegression + BERT)

Evaluate pure Logistic Regression + E5-BERT embeddings (no hierarchy, no DS reasoning).

**What it does:**
- Trains classifier on Banking77 or CLINC150
- Evaluates on test set (all 3,080 samples) /(4500 samples from CLINC150)
- Computes: Accuracy, F1, Precision, Recall, Confidence stats
- Establishes baseline for comparison with DS system

**Output:** `vanilla_metrics.json`, `vanilla_predictions.csv`

In [49]:
# Force reload modules to get latest code changes
import sys
import importlib
import importlib.util
import json
import numpy as np
from pathlib import Path

if 'src.models.embeddings' in sys.modules:
    importlib.reload(sys.modules['src.models.embeddings'])
if 'scripts.training.train' in sys.modules:
    importlib.reload(sys.modules['scripts.training.train'])

# Reload Dropbox modules so .env path fix is always active
for _mod in ['src.utils.dropbox_integration', 'src.utils.dropbox_utils', 'src.utils.dropbox_saver']:
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

from scripts.training.train import train_model

eval_module_path = Path(BASE_DIR) / "scripts" / "evaluation" / "evaluate_vanilla_baseline.py"
spec = importlib.util.spec_from_file_location("evaluate_vanilla_baseline", eval_module_path)
evb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(evb)
evaluate_vanilla_baseline = evb.evaluate_vanilla_baseline

print("="*60)
print("STEP 0: VANILLA BASELINE EVALUATION")
print("="*60)

# ── Step 0.1: Train model ────────────────────────────────────────────────────
print(f"\n[0.1] Training vanilla baseline model for '{DATASET}'...")
train_results = train_model(
    dataset=DATASET,
    classifier_type='logistic',
    embedding_model='intfloat/e5-base',
    output_dir=EXPERIMENTS_DIR,
    experiment_name=f"{DATASET}_logistic",
    hierarchy_file=None,
    intents_file=None,
    max_iter=1000,
    batch_size=64,
    train_samples=TRAIN_SAMPLES,
    eval_samples=None
)

classifier  = train_results['classifier']
embedder    = train_results['embedder']
data_loader = train_results['data_loader']

# ── Step 0.2: Upload freshly trained model to Dropbox ───────────────────────
print("\n[0.2] Uploading trained model to Dropbox...")
try:
    from src.utils.dropbox_utils import upload_model
    _model_path = train_results.get('model_path') or (EXPERIMENTS_DIR / f"{DATASET}_logistic_model.pkl")
    upload_model(Path(_model_path), overwrite=True)
except Exception as _e:
    print(f"   ⚠️  Dropbox upload skipped: {_e}")

# ── Step 0.3: Evaluate on test set ──────────────────────────────────────────
print("\n[0.3] Evaluating on test set...")
vanilla_output_dir = RESULTS_DIR / "vanilla"
vanilla_output_dir.mkdir(parents=True, exist_ok=True)

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score as _f1

metrics = evaluate_vanilla_baseline(
    classifier=classifier,
    embedder=embedder,
    intent_names=data_loader.intent_names,
    dataset_name=DATASET,
    num_samples=EVAL_SAMPLES,
    batch_size=64,
    output_dir=vanilla_output_dir
)

print(f"\n✅ STEP 0 Complete!")
print(f"   Accuracy : {metrics['accuracy']:.4f}")
print(f"   F1 (macro): {metrics['f1_macro']:.4f}")
print(f"   Saved to : {vanilla_output_dir}")


STEP 0: VANILLA BASELINE EVALUATION

[0.1] Training vanilla baseline model for 'banking77'...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]


[0.2] Uploading trained model to Dropbox...
✅ Dropbox already up-to-date: /ds_project_models/banking77_logistic_model.pkl (0.5 MB) — skipped

[0.3] Evaluating on test set...


Batches:   0%|          | 0/49 [00:00<?, ?it/s]


✅ STEP 0 Complete!
   Accuracy : 0.8766
   F1 (macro): 0.8728
   Saved to : /home/kudzai/projects/DS_Project/results/banking77/workflow_demo/vanilla


## STEP 1: Extract Belief Values (DS Baseline, Default Thresholds)

Extract belief distributions from DS system using DEFAULT thresholds.

**What it does:**
- Initialize DS system with DEFAULT confidence thresholds (Leaf: 0.3, Parent: 0.5, Root: 0.7)
- For each query: `compute_mass_function()` → `compute_belief()` for all intents
- Extract per-intent belief scores (no clarifications triggered at this stage)
- Collects belief data to prepare for optimal threshold computation

**Why default thresholds?**
- Gives us raw belief distributions independent of optimization
- STEP 2 will analyze these beliefs to find better per-intent thresholds
- Ensures we're optimizing based on actual belief values, not classifier confidence

**Output:** `banking77_beliefs.csv` (per-intent beliefs for all 3,080 test queries)

In [15]:
# STEP 1: Belief Extraction
import importlib
import scripts.ds_preparation.extract_beliefs
importlib.reload(scripts.ds_preparation.extract_beliefs)
from scripts.ds_preparation.extract_beliefs import extract_beliefs_from_data

from src.models.embeddings import IntentEmbeddings
from src.models.ds_mass_function import DSMassFunction
from config.hierarchy_loader import load_hierarchy_from_json, load_hierarchical_intents_from_json
import pandas as pd

print("="*60)
print("STEP 1: BELIEF EXTRACTION")
print("="*60)

# Load configuration files
config_file = CONFIG_DIR / "hierarchies" / f"{DATASET}_hierarchy.json"
intents_file = CONFIG_DIR / "hierarchies" / f"{DATASET}_intents.json"

print("\nLoading hierarchy and intents...")
hierarchy = load_hierarchy_from_json(str(config_file))
hierarchical_intents = load_hierarchical_intents_from_json(str(intents_file))

# Create intent embeddings
print("Creating intent embeddings...")
intent_embeddings = IntentEmbeddings(hierarchical_intents, embedder=embedder)

# Initialize DS Mass Function (WITHOUT thresholds for belief extraction)
print("Initializing DS Mass Function...")
ds_calculator = DSMassFunction(
    intent_embeddings=intent_embeddings.get_all_embeddings(),
    hierarchy=hierarchy,
    classifier=classifier,
    custom_thresholds=None,  # No thresholds for belief extraction
    enable_belief_tracking=False  # Disable tracking for efficiency
)

# Extract beliefs
print("\nExtracting belief values (no clarifications)...")
results = extract_beliefs_from_data(
    data_loader=data_loader,
    ds_calculator=ds_calculator,
    dataset_name=DATASET,
    split='test',
    num_samples=EVAL_SAMPLES,
    n_chunks=4,
    output_dir=RESULTS_DIR
)

# Save results
print("\nSaving results...")
beliefs_df = pd.DataFrame(results)
beliefs_file = RESULTS_DIR / f"{DATASET}_beliefs.csv"
beliefs_df.to_csv(beliefs_file, index=False)

print(f"\n✅ STEP 1 Complete!")
print(f"   Total samples: {len(beliefs_df)}")
print(f"   Number of intents: {len(beliefs_df.columns) - 2}")
print(f"   Saved to: {beliefs_file}")


STEP 1: BELIEF EXTRACTION

Loading hierarchy and intents...
Creating intent embeddings...
Initializing DS Mass Function...

Extracting belief values (no clarifications)...


Chunk 4: 100%|██████████| 1125/1125 [00:08<00:00, 132.47it/s]



Saving results...

✅ STEP 1 Complete!
   Total samples: 4500
   Number of intents: 160
   Saved to: /home/kudzai/projects/DS_Project/results/clinc150/workflow_demo/clinc150_beliefs.csv


## STEP 2: Threshold Computation

Analyze belief distributions and compute optimal thresholds per intent.

**What it does:**
- Analyze beliefs.csv
- Find threshold that maximizes F1 for each intent
- Balance precision and recall

**Output:** `optimal_thresholds.json`

In [16]:
# STEP 2: Threshold Computation
from scripts.ds_preparation.compute_thresholds import compute_optimal_thresholds, add_ancestor_labels
import json

print("="*60)
print("STEP 2: THRESHOLD COMPUTATION")
print("="*60)

# Load beliefs DataFrame (already loaded from STEP 1)
print(f"\nLoaded beliefs with {len(beliefs_df)} samples")

# Add ancestor-aware ground truth labels
print("Adding ancestor-aware ground truth labels...")
beliefs_df_with_labels = add_ancestor_labels(beliefs_df, hierarchy)

# Identify intent columns (those without 'is_correct_' prefix)
intent_columns = [
    col for col in beliefs_df_with_labels.columns
    if not col.startswith('is_correct_') and col not in ['query', 'true_intent']
]
print(f"Found {len(intent_columns)} intent columns")

# Compute optimal thresholds
print("Computing optimal thresholds...")
optimal_results = compute_optimal_thresholds(
    df=beliefs_df_with_labels,
    intent_columns=intent_columns
)

# Save results
thresholds_file = RESULTS_DIR / f"{DATASET}_optimal_thresholds.json"
with open(thresholds_file, 'w') as f:
    json.dump(optimal_results, f, indent=4)

print(f"\n✅ STEP 2 Complete!")
print(f"   Number of intents: {len(optimal_results)}")
print(f"   Saved to: {thresholds_file}")


STEP 2: THRESHOLD COMPUTATION

Loaded beliefs with 4500 samples
Adding ancestor-aware ground truth labels...
Found 160 intent columns
Computing optimal thresholds...


Processing intents: 100%|██████████| 160/160 [00:10<00:00, 15.45it/s]


✅ STEP 2 Complete!
   Number of intents: 160
   Saved to: /home/kudzai/projects/DS_Project/results/clinc150/workflow_demo/clinc150_optimal_thresholds.json


## STEP 3: DS Evaluation WITH Optimal Thresholds

Evaluate DS system using optimal thresholds computed in STEP 2.

**What it does:**
- Load optimal thresholds from STEP 2 (`banking77_optimal_thresholds.json`)
- Initialize DS system with these custom thresholds (NOT default)
- LLM-based user simulation (CustomerAgent) provides clarifications
- For each query: clarify → combine masses → evaluate hierarchy
- Measure: Accuracy, F1, average clarifications, belief progression

**Key difference from STEP 1:**
- STEP 1 used default thresholds → many clarifications needed
- STEP 3 uses optimal thresholds → selective, targeted clarifications

**Output:** `ds_predictions.csv` with full interaction logs and belief progression

In [17]:
from src.agents.customer_agent import CustomerAgent
from config.threshold_loader import load_thresholds_from_json
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import json

print("="*60)
print("STEP 3: DS EVALUATION WITH OPTIMAL THRESHOLDS")
print("="*60)

# Toggle belief logging for explainability analysis (STEP 9)
TRACK_BELIEF_LOGS = True

# Initialize LLM-based customer agent
print("\nInitializing LLM-based customer agent...")
customer_agent = CustomerAgent(rate_limit_delay=0.1)  # 0.1s for faster testing

# Get test data
test_texts, test_labels, _ = data_loader.get_split_data('test')

# Show sample data
sample_data = [{'text': test_texts[i], 'label': test_labels[i]} for i in range(min(10, len(test_texts)))]
print(f"\nSample of test data (first 10 samples):")
print(sample_data)

# Load optimal thresholds using the utility function
print(f"\nLoading optimal thresholds from: {thresholds_file}")
optimal_thresholds = load_thresholds_from_json(str(thresholds_file))
print(f"Loaded {len(optimal_thresholds)} custom thresholds")
print(f"Sample thresholds: {list(optimal_thresholds.items())[:5]}")

# Create ds_calculator WITH optimal thresholds
ds_calculator = DSMassFunction(
    intent_embeddings=intent_embeddings.get_all_embeddings(),
    hierarchy=hierarchy,
    classifier=classifier,
    custom_thresholds=optimal_thresholds,  # Use optimal thresholds from STEP 2 (as dict!)
    enable_belief_tracking=TRACK_BELIEF_LOGS
)
ds_calculator.customer_agent_callback = customer_agent

# Prepare output locations
output_dir = RESULTS_DIR / "ds_evaluation"
output_dir.mkdir(parents=True, exist_ok=True)

belief_logs_dir = output_dir / "belief_logs"
if TRACK_BELIEF_LOGS:
    belief_logs_dir.mkdir(parents=True, exist_ok=True)

# Evaluate with LLM simulation
print("\nEvaluating DS with optimal thresholds (LLM-based user simulation)...")
results = []
interaction_count = 0

for idx, (text, true_intent) in enumerate(zip(test_texts, test_labels)):
    # Reset conversation
    ds_calculator.conversation_history = []
    ds_calculator.clear_belief_history()
    
    # Compute initial mass and get prediction
    initial_mass = ds_calculator.compute_mass_function(text)
    DS_prediction = ds_calculator.evaluate_with_clarifications(initial_mass)
    
    # Extract prediction
    if DS_prediction:
        pred_intent, confidence = DS_prediction
    else:
        pred_intent, confidence = "unknown", 0.0
    
    # Store result
    result = {
        'query': text,
        'true_intent': true_intent,
        'predicted_intent': pred_intent,
        'confidence': confidence,
        'interaction': '\n'.join(ds_calculator.conversation_history)
    }
    results.append(result)

    # Save belief log for explainability analysis (query_1_belief_log.json, ...)
    if TRACK_BELIEF_LOGS:
        belief_log_file = belief_logs_dir / f"query_{idx + 1}_belief_log.json"
        ds_calculator.save_belief_log(str(belief_log_file))
    
    # Print conversation
    if ds_calculator.conversation_history:
        interaction_count += 1
        print(f"\n--- Interaction {interaction_count} ---")
        print(f"Query: {text}")
        print(f"True intent: {true_intent}")
        print(f"Predicted: {pred_intent}")
        for turn in ds_calculator.conversation_history:
            print(turn)
    
    # Progress indicator
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_texts)} samples...")

# Save results to CSV
results_df = pd.DataFrame(results)
results_file = output_dir / f"{DATASET}_predictions.csv"
results_df.to_csv(results_file, index=False)

print(f"\n{'='*60}")
print(f"✅ EVALUATION COMPLETE!")
print(f"   Total samples: {len(results)}")
print(f"   Samples with clarifications: {interaction_count}")
if TRACK_BELIEF_LOGS:
    print(f"   Belief logs saved: {len(results)}")
    print(f"   Belief log directory: {belief_logs_dir}")
else:
    print("   Belief logs: disabled (TRACK_BELIEF_LOGS=False)")
print(f"   Saved to: {results_file}")
print(f"{'='*60}")

# ============================================================================
# ANALYSIS 
# ============================================================================

print("\n" + "="*60)
print("ANALYZING RESULTS")
print("="*60)

# Load predictions and add analysis columns
df = pd.DataFrame(results)
df['num_clarifications'] = df['interaction'].fillna('').str.count('Chatbot:')
df['is_correct'] = df['true_intent'] == df['predicted_intent']
df['has_clarifications'] = df['num_clarifications'] > 0

# Calculate metrics
print("\n" + "="*60)
print("OVERALL METRICS")
print("="*60)
accuracy = accuracy_score(df['true_intent'], df['predicted_intent'])
f1 = f1_score(df['true_intent'], df['predicted_intent'], average='macro', zero_division=0)
avg_clarifications = df['num_clarifications'].mean()
samples_with_clarif = df['has_clarifications'].sum()
clarification_rate = (samples_with_clarif / len(df)) * 100

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 (macro): {f1:.4f}")
print(f"Avg clarifications per query: {avg_clarifications:.2f}")
print(f"Queries with clarifications: {samples_with_clarif}/{len(df)} ({clarification_rate:.1f}%)")

# Analyze clarification effectiveness
print("\n" + "="*60)
print("CLARIFICATION EFFECTIVENESS")
print("="*60)

clarif_df = df[df['has_clarifications']]
if len(clarif_df) > 0:
    clarif_accuracy = (clarif_df['is_correct']).mean()
    print(f"Accuracy on queries with clarifications: {clarif_accuracy:.4f}")
    print(f"Total clarification turns: {clarif_df['num_clarifications'].sum()}")
    print(f"Avg turns per clarified query: {clarif_df['num_clarifications'].mean():.2f}")
else:
    print("No clarifications needed!")

# Save comprehensive analysis
analysis_file = output_dir / "analysis.json"
analysis_data = {
    'accuracy': float(accuracy),
    'f1_macro': float(f1),
    'avg_clarifications': float(avg_clarifications),
    'samples_with_clarifications': int(samples_with_clarif),
    'clarification_rate_percent': float(clarification_rate),
    'total_samples': len(df),
    'total_clarification_turns': int(clarif_df['num_clarifications'].sum() if len(clarif_df) > 0 else 0),
    'accuracy_on_clarified_queries': float((clarif_df['is_correct']).mean()) if len(clarif_df) > 0 else None
}

with open(analysis_file, 'w') as f:
    json.dump(analysis_data, f, indent=2)

print(f"\n{'='*60}")
print(f"✅ ANALYSIS COMPLETE!")
print(f"   Full results saved to: {results_file}")
print(f"   Analysis saved to: {analysis_file}")
print(f"{'='*60}")

STEP 3: DS EVALUATION WITH OPTIMAL THRESHOLDS

Initializing LLM-based customer agent...

Sample of test data (first 10 samples):
[{'text': 'how would you say fly in italian', 'label': 'translate'}, {'text': "what's the spanish word for pasta", 'label': 'translate'}, {'text': 'how would they say butter in zambia', 'label': 'translate'}, {'text': 'how do you say fast in spanish', 'label': 'translate'}, {'text': "what's the word for trees in norway", 'label': 'translate'}, {'text': 'how does one say wonderful in german', 'label': 'translate'}, {'text': 'how do they say tacos in mexico', 'label': 'translate'}, {'text': 'how would one say cruiser in china', 'label': 'translate'}, {'text': "what's the french word you use for potato", 'label': 'translate'}, {'text': 'what would the word for grass be in finland', 'label': 'translate'}]

Loading optimal thresholds from: /home/kudzai/projects/DS_Project/results/clinc150/workflow_demo/clinc150_optimal_thresholds.json
Loaded 160 custom thresholds


In [18]:
# DEBUG: Check if interactions were actually saved
import pandas as pd

# Define the output directory from STEP 3
predictions_output_dir = RESULTS_DIR / "ds_evaluation"
predictions_file = predictions_output_dir / f"{DATASET}_predictions.csv"

if predictions_file.exists():
    df = pd.read_csv(predictions_file)
    print(f"Loaded predictions CSV: {len(df)} rows")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nSample rows with interaction column:")
    
    # Show non-empty interactions
    has_interactions = df[df['interaction'].notna() & (df['interaction'].str.len() > 0)]
    print(f"Rows with interactions: {len(has_interactions)}")
    
    if len(has_interactions) > 0:
        print(f"\nFirst interaction:")
        print(has_interactions.iloc[0]['interaction'])
    else:
        print("\nNo interactions found in CSV!")
        print(f"\nFirst 3 rows:")
        print(df.head(3))
else:
    print(f"❌ Predictions file not found: {predictions_file}")
    print("Make sure to run STEP 3 (cell 11) first!")


Loaded predictions CSV: 4500 rows
Columns: ['query', 'true_intent', 'predicted_intent', 'confidence', 'interaction']

Sample rows with interaction column:
Rows with interactions: 4500

First interaction:
User: how would you say fly in italian


==========================================================================================================================

## STEP 4: Statistical Comparison (McNemar Test)

Compare DS system with clarifications (STEP 3) vs Vanilla Baseline (STEP 0) using **McNemar's exact test**.

**Purpose:**
- Tests whether DS improvements are **statistically significant**
- McNemar test designed for **paired binary outcomes** (correct/incorrect on same test samples)
- Accounts for **correlated predictions** (both models tested on identical queries)

**Hypotheses:**
- **H0 (Null)**: DS and Baseline have equal error rates (no difference)
- **H1 (Alternative)**: DS has significantly different (better or worse) error rate

**Contingency Table:**
```
                    DS Correct    DS Wrong
Baseline Correct         -            b
Baseline Wrong           c            -
```
- **b**: Baseline correct, DS wrong (regression cases)
- **c**: Baseline wrong, DS correct (improvement cases)

**Interpretation:**
- **p < 0.05**: Statistically significant difference (reject H0)
- **c > b**: DS improves over baseline
- **b > c**: DS regresses from baseline

**Output:** McNemar statistic, p-value, contingency breakdown

In [19]:
import ast
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

print("="*60)
print("STEP 4: STATISTICAL COMPARISON (McNEMAR TEST)")
print("="*60)

# ============================================================================
# AUTO-DETECT VANILLA BASELINE PREDICTIONS (multiple candidate paths)
# ============================================================================
vanilla_candidates = [
    RESULTS_DIR / "vanilla" / f"{DATASET}_vanilla_predictions.csv",  # STEP 0 default
    RESULTS_DIR / "vanilla" / f"{DATASET}_predictions.csv",
    EXPERIMENTS_DIR / "vanilla" / f"{DATASET}_vanilla_predictions.csv",
    EXPERIMENTS_DIR / "vanilla" / f"{DATASET}_predictions.csv",
    BASE_DIR / "results" / DATASET / "vanilla" / "predictions.csv",
    BASE_DIR / "experiments" / DATASET / "vanilla_predictions.csv",
]
vanilla_file = next((p for p in vanilla_candidates if p.exists()), None)
if vanilla_file is None:
    print(f"❌ Vanilla baseline predictions not found!")
    print(f"   Searched: {[str(p) for p in vanilla_candidates]}")
    print(f"   Run STEP 0 first!")
else:
    print(f"✓ Found vanilla predictions: {vanilla_file}")

# ============================================================================
# AUTO-DETECT DS PREDICTIONS (multiple candidate paths)
# ============================================================================
ds_candidates = [
    RESULTS_DIR / "ds_evaluation" / f"{DATASET}_predictions.csv",
    RESULTS_DIR / "ds" / f"{DATASET}_predictions.csv",
    BASE_DIR / "results" / DATASET / f"{DATASET}_ds_predictions.csv",
    BASE_DIR / "outputs" / "evaluation" / f"{DATASET}_predictions.csv",
]
ds_file = next((p for p in ds_candidates if p.exists()), None)
if ds_file is None:
    print(f"❌ DS predictions not found!")
    print(f"   Searched: {[str(p) for p in ds_candidates]}")
    print(f"   Run STEP 3 first!")
else:
    print(f"✓ Found DS predictions: {ds_file}")

# ============================================================================
# PROCEED ONLY IF BOTH FILES FOUND
# ============================================================================
if vanilla_file is not None and ds_file is not None:
    df_vanilla = pd.read_csv(vanilla_file)
    df_ds = pd.read_csv(ds_file)

    print(f"\nLoaded {len(df_vanilla)} vanilla predictions")
    print(f"Loaded {len(df_ds)} DS predictions")

    # --------------------------------------------------------------------------
    # PARSE PREDICTION FORMATS: handles string, tuple ("intent", conf), list
    # --------------------------------------------------------------------------
    def parse_prediction_cell(cell):
        if pd.isna(cell):
            return None
        if isinstance(cell, str):
            try:
                parsed = ast.literal_eval(cell)
                if isinstance(parsed, (list, tuple)) and len(parsed) > 0:
                    return str(parsed[0])
                return str(parsed)
            except (ValueError, SyntaxError):
                return str(cell)
        if isinstance(cell, (list, tuple)) and len(cell) > 0:
            return str(cell[0])
        return str(cell)

    # --------------------------------------------------------------------------
    # GROUND TRUTH
    # --------------------------------------------------------------------------
    if 'true_intent' not in df_vanilla.columns or 'true_intent' not in df_ds.columns:
        print("❌ Missing 'true_intent' column in one or both files!")
    else:
        true_labels_vanilla = df_vanilla['true_intent'].astype(str).values
        true_labels_ds     = df_ds['true_intent'].astype(str).values

        # --------------------------------------------------------------------------
        # EXTRACT VANILLA PREDICTIONS (flexible column names)
        # --------------------------------------------------------------------------
        if 'predicted_intent' in df_vanilla.columns:
            vanilla_preds = df_vanilla['predicted_intent'].astype(str).values
        elif 'prediction' in df_vanilla.columns:
            vanilla_preds = df_vanilla['prediction'].apply(parse_prediction_cell).values
        elif 'baseline_prediction' in df_vanilla.columns:
            vanilla_preds = df_vanilla['baseline_prediction'].astype(str).values
        else:
            print(f"❌ Could not find vanilla predictions! Available columns: {df_vanilla.columns.tolist()}")
            vanilla_preds = None

        # --------------------------------------------------------------------------
        # EXTRACT DS PREDICTIONS (flexible column names)
        # --------------------------------------------------------------------------
        if 'predicted_intent' in df_ds.columns:
            ds_preds = df_ds['predicted_intent'].astype(str).values
        elif 'prediction' in df_ds.columns:
            ds_preds = df_ds['prediction'].apply(parse_prediction_cell).values
        elif 'ds_prediction' in df_ds.columns:
            ds_preds = df_ds['ds_prediction'].apply(parse_prediction_cell).values
        else:
            print(f"❌ Could not find DS predictions! Available columns: {df_ds.columns.tolist()}")
            ds_preds = None

        # --------------------------------------------------------------------------
        # VALIDATION & McNEMAR TEST
        # --------------------------------------------------------------------------
        if vanilla_preds is not None and ds_preds is not None:
            if len(vanilla_preds) != len(ds_preds):
                print(f"❌ Sample size mismatch: Vanilla={len(vanilla_preds)}, DS={len(ds_preds)}")
                print("   Both must be evaluated on the same test set!")
            elif not np.array_equal(true_labels_vanilla, true_labels_ds):
                print("❌ Ground truth mismatch! Vanilla and DS evaluated on different test sets.")
            else:
                true_labels = true_labels_vanilla

                vanilla_correct = vanilla_preds == true_labels
                ds_correct      = ds_preds      == true_labels

                # McNemar contingency counts
                b = np.sum(vanilla_correct & ~ds_correct)   # Vanilla☑ DS✗ → REGRESSION
                c = np.sum(~vanilla_correct & ds_correct)   # Vanilla✗ DS☑ → IMPROVEMENT
                both_correct = np.sum(vanilla_correct & ds_correct)
                both_wrong   = np.sum(~vanilla_correct & ~ds_correct)

                contingency_table = np.array([[0, b], [c, 0]])
                result = mcnemar(contingency_table, exact=True)

                # Display
                print("\n" + "="*60)
                print("CONTINGENCY BREAKDOWN")
                print("="*60)
                print(f"Both models correct:        {both_correct:4d} ({both_correct/len(true_labels)*100:.1f}%)")
                print(f"Both models wrong:          {both_wrong:4d}   ({both_wrong/len(true_labels)*100:.1f}%)")
                print(f"Vanilla correct, DS wrong:  {b:4d}   ({b/len(true_labels)*100:.1f}%) [REGRESSION]")
                print(f"Vanilla wrong, DS correct:  {c:4d}   ({c/len(true_labels)*100:.1f}%) [IMPROVEMENT]")
                print(f"\nNet improvement: {c - b:+d} samples ({(c-b)/len(true_labels)*100:+.1f}%)")

                print("\n" + "="*60)
                print("McNEMAR EXACT TEST RESULTS")
                print("="*60)
                print(f"Test statistic: {result.statistic}")
                print(f"p-value:        {result.pvalue:.6f}")

                alpha = 0.05
                if result.pvalue < alpha:
                    if c > b:
                        print(f"\n✅ CONCLUSION (α={alpha}): DS system is SIGNIFICANTLY BETTER than baseline (p={result.pvalue:.4f})")
                    else:
                        print(f"\n❌ CONCLUSION (α={alpha}): DS system is SIGNIFICANTLY WORSE than baseline (p={result.pvalue:.4f})")
                else:
                    print(f"\n⚠️  CONCLUSION (α={alpha}): No statistically significant difference (p={result.pvalue:.4f})")
                    print("    DS and Vanilla have comparable performance.")

                # Save results
                mcnemar_file = RESULTS_DIR / "mcnemar_test_results.json"
                mcnemar_data = {
                    'dataset': DATASET,
                    'vanilla_file': str(vanilla_file),
                    'ds_file': str(ds_file),
                    'vanilla_accuracy': float(vanilla_correct.mean()),
                    'ds_accuracy': float(ds_correct.mean()),
                    'improvement_delta': float(ds_correct.mean() - vanilla_correct.mean()),
                    'both_correct': int(both_correct),
                    'both_wrong': int(both_wrong),
                    'vanilla_only_correct': int(b),
                    'ds_only_correct': int(c),
                    'net_improvement': int(c - b),
                    'mcnemar_statistic': float(result.statistic),
                    'mcnemar_pvalue': float(result.pvalue),
                    'significant_at_0.05': bool(result.pvalue < 0.05),
                    'conclusion': 'DS better' if (result.pvalue < 0.05 and c > b) else (
                                  'DS worse'  if (result.pvalue < 0.05 and b > c) else 'No difference')
                }
                with open(mcnemar_file, 'w') as f:
                    json.dump(mcnemar_data, f, indent=2)

                print(f"\n{'='*60}")
                print(f"✅ McNEMAR TEST COMPLETE!")
                print(f"   Results saved to: {mcnemar_file}")
                print(f"{'='*60}")


STEP 4: STATISTICAL COMPARISON (McNEMAR TEST)
✓ Found vanilla predictions: /home/kudzai/projects/DS_Project/results/clinc150/workflow_demo/vanilla/clinc150_vanilla_predictions.csv
✓ Found DS predictions: /home/kudzai/projects/DS_Project/results/clinc150/workflow_demo/ds_evaluation/clinc150_predictions.csv

Loaded 4500 vanilla predictions
Loaded 4500 DS predictions

CONTINGENCY BREAKDOWN
Both models correct:        4196 (93.2%)
Both models wrong:           154   (3.4%)
Vanilla correct, DS wrong:    63   (1.4%) [REGRESSION]
Vanilla wrong, DS correct:    87   (1.9%) [IMPROVEMENT]

Net improvement: +24 samples (+0.5%)

McNEMAR EXACT TEST RESULTS
Test statistic: 63.0
p-value:        0.060027

⚠️  CONCLUSION (α=0.05): No statistically significant difference (p=0.0600)
    DS and Vanilla have comparable performance.

✅ McNEMAR TEST COMPLETE!
   Results saved to: /home/kudzai/projects/DS_Project/results/clinc150/workflow_demo/mcnemar_test_results.json


==================================================================================================================================

## STEP 5: Select User Study Queries

Select challenging queries from STEP 3 results for real human validation.

**What it does:**
- **Validates ALL predictions against ground truth labels** (`is_correct = predicted_intent == true_intent`)
- Focuses on **high-interaction queries only** (>= 2 clarification turns)
- Splits into **2 groups** by **CORRECTNESS** (not confidence):
  - **Successful**: Model predicted CORRECTLY (matches true_intent)
  - **Problematic**: Model predicted INCORRECTLY (doesn't match true_intent)
- Further splits by **interaction level**:
  - **Level 1**: Exactly 2 turns
  - **Level 2**: More than 2 turns
- Creates **4 balanced categories**:
  1. `successful_level1` - CORRECT predictions with 2 turns
  2. `successful_level2` - CORRECT predictions with >2 turns
  3. `problematic_level1` - INCORRECT predictions with 2 turns
  4. `problematic_level2` - INCORRECT predictions with >2 turns

**Why validate against ground truth?**
- High confidence ≠ correct prediction (model can be confidently wrong!)
- We need to identify where the system actually fails vs. where it succeeds
- Confidence is logged for analysis, but correctness drives selection

**Why focus on high-interaction queries?**
- These queries required clarification dialogue (not single-shot classification)
- Represent edge cases where system needed multiple turns
- Most insightful for human validation: tests multi-turn clarification effectiveness
- Separating by interaction level reveals: simple clarifications (2 turns) vs complex negotiations (>2 turns)

**Output:** `selected_queries_for_user_study.csv` with balanced sampling across 4 categories, each validated against true dataset labels

In [20]:
import sys
import importlib
sys.path.insert(0, str(BASE_DIR))

# Import and reload QuerySelector
import src.utils.query_selector as qs
importlib.reload(qs)
from src.utils.query_selector import QuerySelector
import pandas as pd

print("="*60)
print("STEP 5: SELECT USER STUDY QUERIES")
print("="*60)

# Load predictions from STEP 3
predictions_file = RESULTS_DIR / "ds_evaluation" / f"{DATASET}_predictions.csv"
if not predictions_file.exists():
    print("❌ Predictions file not found! Run STEP 3 first.")
else:
    results_df = pd.read_csv(predictions_file)
    print(f"Loaded {len(results_df)} predictions from STEP 3")
    
    # Show full dataset distribution first
    selector_temp = QuerySelector(min_interactions=2)
    results_with_meta = selector_temp._preprocess(results_df)
    
    high_interaction = results_with_meta[results_with_meta['num_interactions'] >= 2]
    total_successful = high_interaction['is_correct'].sum()
    total_problematic = len(high_interaction) - total_successful
    
    print(f"\n📊 Full Dataset Distribution (high-interaction only, ≥2 turns):")
    print(f"   Total high-interaction: {len(high_interaction)} queries")
    print(f"   • Successful (correct):   {total_successful} ({total_successful/len(high_interaction)*100:.1f}%)")
    print(f"   • Problematic (incorrect): {total_problematic} ({total_problematic/len(high_interaction)*100:.1f}%)")
    
    # Initialize QuerySelector
    selector = QuerySelector(min_interactions=2)
    
    # Select queries by interaction level and correctness
    print(f"\nSelecting {HUMAN_STUDY_SAMPLES} queries for user study...")
    selected_queries = selector.select_by_interaction_levels(
        results_df=results_df,
        max_samples=HUMAN_STUDY_SAMPLES
    )
    
    if not selected_queries.empty:
        summary = selector.generate_study_summary(selected_queries)
        
        # Show key results
        correct = selected_queries['is_correct'].sum()
        incorrect = len(selected_queries) - correct
        
        print("\n" + "="*60)
        print("SELECTED SUBSET")
        print("="*60)
        print(f"Total selected: {len(selected_queries)} queries")
        print(f"  Correct predictions:   {correct} ({correct/len(selected_queries)*100:.1f}%)")
        print(f"  Incorrect predictions: {incorrect} ({incorrect/len(selected_queries)*100:.1f}%)")
        print(f"  Avg interactions: {summary['avg_interactions']:.2f} turns")
        print(f"  Avg confidence: {summary['avg_confidence']:.4f}")
        print(f"\nCategories:")
        for cat, count in summary['categories'].items():
            print(f"  • {cat}: {count}")
        
        # Save files
        diagnostics_file = USER_STUDY_DIR / "selection_pool_diagnostics.csv"
        selected_file = USER_STUDY_DIR / "selected_queries_for_user_study.csv"
        
        diagnostics_df = pd.DataFrame([{
            "dataset": DATASET,
            "strategy": "interaction_levels",
            "total_dataset_high_interaction": len(high_interaction),
            "total_dataset_successful": int(total_successful),
            "total_dataset_problematic": int(total_problematic),
            "selected_correct": int(correct),
            "selected_incorrect": int(incorrect),
            **summary
        }])
        diagnostics_file.parent.mkdir(parents=True, exist_ok=True)
        diagnostics_df.to_csv(diagnostics_file, index=False)
        
        selected_file.parent.mkdir(parents=True, exist_ok=True)
        selected_queries.to_csv(selected_file, index=False)
        
        # Save a dataset-specific copy for the MERGE STEP (run after both workflows complete)
        dataset_backup = USER_STUDY_DIR / f"{DATASET}_selected_queries.csv"
        selected_queries.to_csv(dataset_backup, index=False)
        
        print(f"\n✅ Saved to: {selected_file}")
        print(f"   Dataset backup: {dataset_backup}")
        print("="*60)
    else:
        print("\n❌ No queries selected!")


STEP 5: SELECT USER STUDY QUERIES
Loaded 4500 predictions from STEP 3

📊 Full Dataset Distribution (high-interaction only, ≥2 turns):
   Total high-interaction: 220 queries
   • Successful (correct):   86 (39.1%)
   • Problematic (incorrect): 134 (60.9%)

Selecting 100 queries for user study...

SELECTED SUBSET
Total selected: 85 queries
  Correct predictions:   50 (58.8%)
  Incorrect predictions: 35 (41.2%)
  Avg interactions: 3.19 turns
  Avg confidence: 0.6565

Categories:
  • successful_level1: 25
  • successful_level2: 25
  • problematic_level2: 25
  • problematic_level1: 10

✅ Saved to: /home/kudzai/projects/DS_Project/outputs/user_study/workflow_demo/selected_queries_for_user_study.csv
   Dataset backup: /home/kudzai/projects/DS_Project/outputs/user_study/workflow_demo/clinc150_selected_queries.csv


==================================================================================================================

## MERGE STEP: Combine Datasets for User Study

Run the cell below **once after completing both the Banking77 and Clinc150 workflows** (STEP 0 → STEP 5 for each).

It merges both sets of selected queries into a single shuffled file that the Streamlit app reads automatically. Each query carries a `dataset` column so the app routes it to the correct DS model behind the scenes — participants see nothing different.

In [24]:
print("="*60)
print("MERGE STEP: COMBINE DUAL-DATASET QUERIES FOR USER STUDY")
print("="*60)
print("Run after completing BOTH Banking77 AND Clinc150 workflows (STEP 0→5)")
print()

dataset_files = {
    'banking77': USER_STUDY_DIR / "banking77_selected_queries.csv",
    'clinc150':  USER_STUDY_DIR / "clinc150_selected_queries.csv",
}
merged_output = USER_STUDY_DIR / "selected_queries_for_user_study.csv"

frames = []
for name, path in dataset_files.items():
    if path.exists():
        df_tmp = pd.read_csv(path)
        df_tmp['dataset'] = name
        frames.append(df_tmp)
        print(f"✓ Loaded {len(df_tmp)} queries from {name}")
    else:
        print(f"⚠️  Not found yet: {path.name}  →  run the {name} workflow (STEP 0→5) first")

if len(frames) == 2:
    merged = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
    merged_output.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(merged_output, index=False)

    b77  = (merged['dataset'] == 'banking77').sum()
    c150 = (merged['dataset'] == 'clinc150').sum()
    print(f"\n{'='*60}")
    print(f"✅ MERGED QUERY FILE READY FOR STREAMLIT")
    print(f"{'='*60}")
    print(f"   Total queries : {len(merged)}")
    print(f"   Banking77     : {b77}")
    print(f"   Clinc150      : {c150}")
    print(f"   Saved to      : {merged_output}")
    print(f"\n   Streamlit will route each query to the correct DS model")
    print(f"   via the 'dataset' column — invisible to participants.")
elif len(frames) == 1:
    name_done = [n for n, p in dataset_files.items() if p.exists()][0]
    merged_output.parent.mkdir(parents=True, exist_ok=True)
    frames[0].to_csv(merged_output, index=False)
    print(f"\n⚠️  Only {name_done} is ready — saved as single-dataset file for now.")
    print(f"   Run the other workflow then re-run this cell to produce the full merge.")
else:
    print("\n❌ No dataset query files found. Run both workflows (STEP 0→5) first.")


MERGE STEP: COMBINE DUAL-DATASET QUERIES FOR USER STUDY
Run after completing BOTH Banking77 AND Clinc150 workflows (STEP 0→5)

✓ Loaded 100 queries from banking77
✓ Loaded 85 queries from clinc150

✅ MERGED QUERY FILE READY FOR STREAMLIT
   Total queries : 185
   Banking77     : 100
   Clinc150      : 85
   Saved to      : /home/kudzai/projects/DS_Project/outputs/user_study/workflow_demo/selected_queries_for_user_study.csv

   Streamlit will route each query to the correct DS model
   via the 'dataset' column — invisible to participants.


==============================================================================================================

## EXPERIMENT SETS: Build 3 Balanced Study Datasets (50% Banking77 / 50% Clinc150)

Creates three query sets of increasing size with a **1:1 banking77-to-clinc150 ratio** (50/50), sampling across the 8 experimental groups:

| Group | Description |
|-------|-------------|
| banking77 / successful_level1 | Correct prediction, exactly 2 turns |
| banking77 / successful_level2 | Correct prediction, >2 turns |
| banking77 / problematic_level1 | Wrong prediction, exactly 2 turns |
| banking77 / problematic_level2 | Wrong prediction, >2 turns |
| clinc150 / successful_level1 | Correct prediction, exactly 2 turns |
| clinc150 / successful_level2 | Correct prediction, >2 turns |
| clinc150 / problematic_level1 | Wrong prediction, exactly 2 turns |
| clinc150 / problematic_level2 | Wrong prediction, >2 turns |

**Three sets produced (50% b77 / 50% clinc150 = 1:1 ratio per group):**

| Set | b77 per group | clinc150 per group | Total b77 | Total clinc150 | Grand total |
|-----|--------------|-------------------|-----------|----------------|--------------|
| `study_set_small.csv`  | 2 | 2 | 8  | 8  | **16** |
| `study_set_medium.csv` | 4 | 4 | 16 | 16 | **32** |
| `study_set_large.csv`  | 6 | 6 | 24 | 24 | **48** |

Run after the MERGE STEP. Streamlit routes each query to the correct DS model automatically via the `dataset` column.
Uploaded to `/ds_project_queries/` root on Dropbox (used by `app_main_hicxai.py`).

In [ ]:
import sys
import pandas as pd
from pathlib import Path

# Always anchor project root from cwd so sys.path is correct regardless of cell run order
_nb_root = Path.cwd()
if _nb_root.name == "notebooks":
    _nb_root = _nb_root.parent
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

# Source: merged file lives in workflow_demo (produced by MERGE STEP)
if 'USER_STUDY_DIR' not in dir():
    USER_STUDY_DIR = _nb_root / "outputs" / "user_study" / "workflow_demo"

# Output: 50/50 sets go into workflow_demo/ (same dir as the source)
# Dropbox upload target: /ds_project_queries/ root
WORKFLOW_DEMO_DIR = _nb_root / "outputs" / "user_study" / "workflow_demo"
WORKFLOW_DEMO_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("EXPERIMENT SETS: 50/50 BALANCED STUDY DATASETS  ->  workflow_demo/")
print("=" * 60)

_merged_file = USER_STUDY_DIR / "selected_queries_for_user_study.csv"
if not _merged_file.exists():
    print("X Merged file not found. Run the MERGE STEP first.")
else:
    _merged = pd.read_csv(_merged_file)

    # Build the 8-group key: dataset + selection_category
    if 'selection_category' not in _merged.columns:
        print("X 'selection_category' column missing -- re-run STEP 5 for both datasets.")
    else:
        # Show what we have
        _merged['_group'] = _merged['dataset'] + ' / ' + _merged['selection_category']
        print("\nAvailable queries per group (from merged file):")
        _counts = _merged.groupby('_group').size().sort_index()
        for grp, n in _counts.items():
            print(f"  {grp:<50} {n:>4} queries")

        # -- helper ----------------------------------------------------------
        def _make_set(df: pd.DataFrame, n_b77: int, n_c150: int, seed: int = 42) -> pd.DataFrame:
            """Sample n_b77 per banking77 group and n_c150 per clinc150 group.

            Gives a 50/50 split when n_b77 == n_c150 (e.g. 2/2, 4/4, 6/6).
            Falls back to all available rows if fewer than requested.
            """
            _quota = {'banking77': n_b77, 'clinc150': n_c150}
            frames = []
            for grp, gdf in df.groupby('_group'):
                ds = grp.split(' / ')[0]
                quota = _quota.get(ds, n_b77)
                available = len(gdf)
                actual = min(quota, available)
                if actual < quota:
                    print(f"  WARNING: {grp}: only {available} available -- taking all {available}")
                frames.append(gdf.sample(n=actual, random_state=seed))
            combined = (
                pd.concat(frames, ignore_index=True)
                  .sample(frac=1, random_state=seed)  # shuffle
                  .reset_index(drop=True)
                  .drop(columns=['_group'])
            )
            return combined

        # -- produce the three sets (50% b77 / 50% clinc150) -----------------
        # Each entry: (n_per_b77_group, n_per_clinc150_group)
        # 4 groups x n_b77 + 4 groups x n_c150 = total
        #   small : 4x2 + 4x2 =  8 +  8 = 16
        #   medium: 4x4 + 4x4 = 16 + 16 = 32
        #   large : 4x6 + 4x6 = 24 + 24 = 48
        _sets = {
            'study_set_small':  (2, 2),
            'study_set_medium': (4, 4),
            'study_set_large':  (6, 6),
        }

        print()
        for _name, (_nb77, _nc150) in _sets.items():
            _df = _make_set(_merged, n_b77=_nb77, n_c150=_nc150)
            _path = WORKFLOW_DEMO_DIR / f"{_name}.csv"
            _df.to_csv(_path, index=False)

            _b77  = (_df['dataset'] == 'banking77').sum()
            _c150 = (_df['dataset'] == 'clinc150').sum()
            _pct_b77  = _b77  / len(_df) * 100
            _pct_c150 = _c150 / len(_df) * 100
            print(f"OK workflow_demo/{_name}.csv  ->  {len(_df):>3} queries  "
                  f"(banking77: {_b77} [{_pct_b77:.0f}%], "
                  f"clinc150: {_c150} [{_pct_c150:.0f}%])")

        # -- upload to Dropbox root /ds_project_queries/ ----------------------
        print()
        try:
            from src.utils.dropbox_saver import upload_file_to_dropbox
            for _name in _sets:
                _local  = WORKFLOW_DEMO_DIR / f"{_name}.csv"
                _remote = f"/ds_project_queries/{_name}.csv"
                upload_file_to_dropbox(str(_local), _remote, overwrite=True)
        except Exception as _e:
            print(f"WARNING: Dropbox upload skipped: {_e}")
            print("   Set DROPBOX_APP_KEY / DROPBOX_APP_SECRET / DROPBOX_REFRESH_TOKEN in .env.")

        print()
        print(f"Files written to: {WORKFLOW_DEMO_DIR}")
        print("Dropbox: /ds_project_queries/study_set_{small,medium,large}.csv")
        print("app_main_hicxai.py reads from /ds_project_queries/ root via STUDY_SET_DROPBOX_FOLDER")
        print("=" * 60)


EXPERIMENT SETS: BALANCED STUDY DATASETS

Available queries per group:
  banking77 / problematic_level1                       25 queries
  banking77 / problematic_level2                       25 queries
  banking77 / successful_level1                        25 queries
  banking77 / successful_level2                        25 queries
  clinc150 / problematic_level1                        10 queries
  clinc150 / problematic_level2                        25 queries
  clinc150 / successful_level1                         25 queries
  clinc150 / successful_level2                         25 queries

✅ study_set_small.csv  →   16 queries  (banking77: 8, clinc150: 8)
✅ study_set_medium.csv  →   40 queries  (banking77: 20, clinc150: 20)
✅ study_set_large.csv  →   80 queries  (banking77: 40, clinc150: 40)

☁️  Uploaded study_set_small.csv → Dropbox:/ds_project_queries/study_set_small.csv
☁️  Uploaded study_set_medium.csv → Dropbox:/ds_project_queries/study_set_medium.csv
☁️  Uploaded study_set_la

## EXPERIMENT SETS (B77-ONLY): Build 3 100% Banking77 Study Datasets

Creates three query sets of increasing size using **only banking77 queries**, sampling across the 4 banking77 experimental groups:

| Group | Description |
|-------|-------------|
| banking77 / successful_level1 | Correct prediction, exactly 2 turns |
| banking77 / successful_level2 | Correct prediction, >2 turns |
| banking77 / problematic_level1 | Wrong prediction, exactly 2 turns |
| banking77 / problematic_level2 | Wrong prediction, >2 turns |

**Three sets produced (100% banking77):**

| Set | queries per group | Total |
|-----|------------------|-------|
| `study_set_small.csv`  | 4 | **16** |
| `study_set_medium.csv` | 8 | **32** |
| `study_set_large.csv`  | 12 | **48** |

Used by `app_main_study.py` (experimental condition). `app_main_hicxai.py` keeps the 50/50 workflow_demo sets (control).

Run after the MERGE STEP. Output directory: `outputs/user_study/study_b77only/`

In [5]:
import sys
import pandas as pd
from pathlib import Path

# Always anchor project root from cwd
_nb_root = Path.cwd()
if _nb_root.name == "notebooks":
    _nb_root = _nb_root.parent
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

# Source: merged file (produced by MERGE STEP)
if 'USER_STUDY_DIR' not in dir():
    USER_STUDY_DIR = _nb_root / "outputs" / "user_study" / "workflow_demo"

# Output: 100% b77 sets into a dedicated directory
STUDY_B77_DIR = _nb_root / "outputs" / "user_study" / "study_b77only"
STUDY_B77_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("EXPERIMENT SETS: 100% BANKING77 STUDY DATASETS  →  study_b77only/")
print("=" * 60)

_merged_file = USER_STUDY_DIR / "selected_queries_for_user_study.csv"
if not _merged_file.exists():
    print("❌ Merged file not found. Run the MERGE STEP first.")
else:
    _merged = pd.read_csv(_merged_file)

    if 'selection_category' not in _merged.columns:
        print("❌ 'selection_category' column missing — re-run STEP 5 for both datasets.")
    else:
        # Filter to banking77 only
        _b77_only = _merged[_merged['dataset'] == 'banking77'].copy()
        _b77_only['_group'] = _b77_only['selection_category']

        print(f"\nAvailable banking77 queries per group:")
        _counts = _b77_only.groupby('_group').size().sort_index()
        for grp, n in _counts.items():
            print(f"  {grp:<50} {n:>4} queries")

        def _make_b77_set(df: pd.DataFrame, n_per_group: int, seed: int = 42) -> pd.DataFrame:
            """Sample n_per_group queries from each banking77 selection_category group."""
            frames = []
            for grp, gdf in df.groupby('_group'):
                available = len(gdf)
                actual = min(n_per_group, available)
                if actual < n_per_group:
                    print(f"  ⚠️  {grp}: only {available} available — taking all {available}")
                frames.append(gdf.sample(n=actual, random_state=seed))
            combined = (
                pd.concat(frames, ignore_index=True)
                  .sample(frac=1, random_state=seed)
                  .reset_index(drop=True)
                  .drop(columns=['_group'])
            )
            return combined

        # small: 4×4=16, medium: 4×8=32, large: 4×12=48
        _sets = {
            'study_set_small':  4,
            'study_set_medium': 8,
            'study_set_large':  12,
        }

        print()
        for _name, _n in _sets.items():
            _df = _make_b77_set(_b77_only, n_per_group=_n)
            _path = STUDY_B77_DIR / f"{_name}.csv"
            _df.to_csv(_path, index=False)
            print(f"✅ study_b77only/{_name}.csv  →  {len(_df):>3} queries  (100% banking77)")

        # Upload to Dropbox under /ds_project_queries/study_b77only/
        print()
        try:
            from src.utils.dropbox_saver import upload_file_to_dropbox
            for _name in _sets:
                _local  = STUDY_B77_DIR / f"{_name}.csv"
                _remote = f"/ds_project_queries/study_b77only/{_name}.csv"
                upload_file_to_dropbox(str(_local), _remote, overwrite=True)
        except Exception as _e:
            print(f"⚠️  Dropbox upload skipped: {_e}")
            print("   Set DROPBOX_APP_KEY / DROPBOX_APP_SECRET / DROPBOX_REFRESH_TOKEN in .env.")

        print()
        print(f"Files written to: {STUDY_B77_DIR}")
        print("app_main_study.py uses STUDY_SET_DIR=outputs/user_study/study_b77only")
        print("app_main_hicxai.py keeps outputs/user_study/workflow_demo (50/50, control)")
        print("=" * 60)


EXPERIMENT SETS: 100% BANKING77 STUDY DATASETS  →  study_b77only/

Available banking77 queries per group:
  problematic_level1                                   25 queries
  problematic_level2                                   25 queries
  successful_level1                                    25 queries
  successful_level2                                    25 queries

✅ study_b77only/study_set_small.csv  →   16 queries  (100% banking77)
✅ study_b77only/study_set_medium.csv  →   32 queries  (100% banking77)
✅ study_b77only/study_set_large.csv  →   48 queries  (100% banking77)

☁️  Uploaded study_set_small.csv → Dropbox:/ds_project_queries/study_b77only/study_set_small.csv
☁️  Uploaded study_set_medium.csv → Dropbox:/ds_project_queries/study_b77only/study_set_medium.csv
☁️  Uploaded study_set_large.csv → Dropbox:/ds_project_queries/study_b77only/study_set_large.csv

Files written to: /home/kudzai/projects/DS_Project/outputs/user_study/study_b77only
app_main_study.py uses STUDY_SET_DIR=ou

## EXPERIMENT SETS (CLINC150-ONLY): Build 3 100% CLINC150 Study Datasets
Creates three query sets of increasing size using ONLY CLINC150 queries.
Used by app_hicxai_clinc_1.py through app_hicxai_clinc_4.py.
Output: outputs/user_study/study_clinc150only/

In [1]:
import sys
import pandas as pd
from pathlib import Path

# Always anchor project root from cwd
_nb_root = Path.cwd()
if _nb_root.name == "notebooks":
    _nb_root = _nb_root.parent
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

if 'USER_STUDY_DIR' not in dir():
    USER_STUDY_DIR = _nb_root / "outputs" / "user_study" / "workflow_demo"

STUDY_CLINC150_DIR = _nb_root / "outputs" / "user_study" / "study_clinc150only"
STUDY_CLINC150_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("EXPERIMENT SETS: 100% CLINC150 STUDY DATASETS  →  study_clinc150only/")
print("=" * 60)

_merged_file = USER_STUDY_DIR / "selected_queries_for_user_study.csv"
if not _merged_file.exists():
    print("❌ Merged file not found. Run the MERGE STEP first.")
else:
    _merged = pd.read_csv(_merged_file)

    if 'selection_category' not in _merged.columns:
        print("❌ 'selection_category' column missing — re-run STEP 5 for both datasets.")
    else:
        _c150_only = _merged[_merged['dataset'] == 'clinc150'].copy()
        _c150_only['_group'] = _c150_only['selection_category']

        print("\nAvailable CLINC150 queries per group:")
        _counts = _c150_only.groupby('_group').size().sort_index()
        for grp, n in _counts.items():
            print(f"  {grp:<50} {n:>4}")

        def _make_clinc150_set(df: pd.DataFrame, n_per_group: int, seed: int = 42) -> pd.DataFrame:
            """Sample n_per_group queries from each selection_category group."""
            frames = []
            for grp, gdf in df.groupby('_group'):
                available = len(gdf)
                actual = min(n_per_group, available)
                if actual < n_per_group:
                    print(f"  ⚠️  {grp}: only {available} available — taking all {available}")
                frames.append(gdf.sample(n=actual, random_state=seed))
            return (
                pd.concat(frames, ignore_index=True)
                  .sample(frac=1, random_state=seed)
                  .reset_index(drop=True)
                  .drop(columns=['_group'])
            )

        # small: 4×4=16, medium: 4×8=32, large: 4×12=48 (may be less if group has <12)
        _sets = {
            'study_set_small':  4,
            'study_set_medium': 8,
            'study_set_large':  12,
        }

        print()
        for _name, _n in _sets.items():
            _df = _make_clinc150_set(_c150_only, n_per_group=_n)
            _path = STUDY_CLINC150_DIR / f"{_name}.csv"
            _df.to_csv(_path, index=False)
            print(f"✅ study_clinc150only/{_name}.csv  →  {len(_df):>3} queries  (100% clinc150)")

        # Upload to Dropbox under /ds_project_queries/study_clinc150only/
        print()
        try:
            from src.utils.dropbox_saver import upload_file_to_dropbox
            for _name in _sets:
                _local  = STUDY_CLINC150_DIR / f"{_name}.csv"
                _remote = f"/ds_project_queries/study_clinc150only/{_name}.csv"
                upload_file_to_dropbox(str(_local), _remote, overwrite=True)
        except Exception as _e:
            print(f"WARNING: Dropbox upload skipped: {_e}")
            print("   Set DROPBOX_APP_KEY / DROPBOX_APP_SECRET / DROPBOX_REFRESH_TOKEN in .env.")

        print()
        print(f"Files written to: {STUDY_CLINC150_DIR}")
        print("Dropbox: /ds_project_queries/study_clinc150only/study_set_{small,medium,large}.csv")
        print("app_hicxai_clinc_1..4.py reads from study_clinc150only/ via STUDY_SET_DIR")
        print("=" * 60)


EXPERIMENT SETS: 100% CLINC150 STUDY DATASETS  →  study_clinc150only/

Available CLINC150 queries per group:
  problematic_level1                                   10
  problematic_level2                                   25
  successful_level1                                    25
  successful_level2                                    25

✅ study_clinc150only/study_set_small.csv  →   16 queries  (100% clinc150)
✅ study_clinc150only/study_set_medium.csv  →   32 queries  (100% clinc150)
  ⚠️  problematic_level1: only 10 available — taking all 10
✅ study_clinc150only/study_set_large.csv  →   46 queries  (100% clinc150)

☁️  Uploaded study_set_small.csv → Dropbox:/ds_project_queries/study_clinc150only/study_set_small.csv
☁️  Uploaded study_set_medium.csv → Dropbox:/ds_project_queries/study_clinc150only/study_set_medium.csv
☁️  Uploaded study_set_large.csv → Dropbox:/ds_project_queries/study_clinc150only/study_set_large.csv

Files written to: /home/kudzai/projects/DS_Project/outputs/user_s

====================================================================================================================================================================================================================

In [23]:
# RECOVERY CELL: Regenerate banking77_selected_queries.csv from existing results
# (Run once if banking77_selected_queries.csv is missing — e.g. STEP 5 ran before backup-save was added)
import pandas as pd
import importlib, sys
from pathlib import Path

_b77_results = BASE_DIR / "results" / "banking77" / "workflow_demo" / "ds_evaluation" / "banking77_predictions.csv"
_b77_backup  = USER_STUDY_DIR / "banking77_selected_queries.csv"

if _b77_backup.exists():
    print(f"✅ banking77_selected_queries.csv already exists ({len(pd.read_csv(_b77_backup))} rows) — nothing to do.")
elif not _b77_results.exists():
    print(f"❌ Banking77 DS predictions not found at:\n   {_b77_results}\n   Run Banking77 STEP 3 first.")
else:
    import src.utils.query_selector as qs
    importlib.reload(qs)
    from src.utils.query_selector import QuerySelector

    _df = pd.read_csv(_b77_results)
    print(f"Loaded {len(_df)} Banking77 predictions → selecting {HUMAN_STUDY_SAMPLES} queries …")

    _sel = QuerySelector(min_interactions=2).select_by_interaction_levels(
        results_df=_df, max_samples=HUMAN_STUDY_SAMPLES
    )

    if _sel.empty:
        print("❌ No queries selected — check Banking77 predictions file.")
    else:
        _b77_backup.parent.mkdir(parents=True, exist_ok=True)
        _sel.to_csv(_b77_backup, index=False)
        correct = _sel['is_correct'].sum()
        print(f"✅ Saved {len(_sel)} queries ({correct} correct / {len(_sel)-correct} problematic)")
        print(f"   → {_b77_backup}")
        print("\nNow re-run the MERGE STEP cell above to produce the combined file.")


Loaded 3080 Banking77 predictions → selecting 100 queries …
✅ Saved 100 queries (50 correct / 50 problematic)
   → /home/kudzai/projects/DS_Project/outputs/user_study/workflow_demo/banking77_selected_queries.csv

Now re-run the MERGE STEP cell above to produce the combined file.


====================================================================================================================================================================================================================

## Phase 5: Comprehensive Comparison

Compare ALL evaluation methods with explainability metrics.

**Compares:**
- Vanilla Baseline (LogisticRegression + BERT)
- DS without thresholds
- DS with optimal thresholds
- LLM simulation results
- Explainability metrics

**Output:**
- Comparison table and plots
- Summary report with key findings
- Faithfulness comparison

In [30]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from pathlib import Path

print("="*70)
print("PHASE 5: COMPREHENSIVE COMPARISON  (Banking77 + Clinc150)")
print("="*70)

_datasets = ['banking77', 'clinc150']
_per_dataset_rows = []
_all_outputs = {}

for _ds in _datasets:
    _rdir         = BASE_DIR / "results" / _ds / "workflow_demo"
    _v_file       = _rdir / "vanilla"       / f"{_ds}_vanilla_predictions.csv"
    _d_file       = _rdir / "ds_evaluation" / f"{_ds}_predictions.csv"
    _a_file       = _rdir / "ds_evaluation" / "analysis.json"
    _m_file       = _rdir / "mcnemar_test_results.json"
    _cmp_dir      = _rdir / "comparison"
    _cmp_dir.mkdir(parents=True, exist_ok=True)

    if not _v_file.exists() or not _d_file.exists():
        print(f"\n⚠️  {_ds}: predictions missing — run STEP 0 + STEP 3 first.")
        continue

    _dfv  = pd.read_csv(_v_file)
    _dfd  = pd.read_csv(_d_file)

    _v_acc = (_dfv['true_intent'].astype(str) == _dfv['predicted_intent'].astype(str)).mean()
    _v_f1  = f1_score(_dfv['true_intent'], _dfv['predicted_intent'], average='macro', zero_division=0)
    _v_conf = _dfv['confidence'].mean() if 'confidence' in _dfv.columns else float('nan')

    _d_acc = (_dfd['true_intent'].astype(str) == _dfd['predicted_intent'].astype(str)).mean()
    _d_f1  = f1_score(_dfd['true_intent'], _dfd['predicted_intent'], average='macro', zero_division=0)
    _d_conf = _dfd['confidence'].mean() if 'confidence' in _dfd.columns else float('nan')

    _ana  = json.load(open(_a_file)) if _a_file.exists() else {}
    _mcn  = json.load(open(_m_file)) if _m_file.exists() else {}

    _clarif_rate = _ana.get('clarification_rate_percent', float('nan'))
    _avg_clarif  = _ana.get('avg_clarifications',         float('nan'))
    _acc_delta   = _d_acc - _v_acc
    _f1_delta    = _d_f1  - _v_f1

    # ── per-dataset comparison table ─────────────────────────────────────────
    print(f"\n{'─'*70}")
    print(f"  {_ds.upper()}")
    print(f"{'─'*70}")
    _tbl = pd.DataFrame([
        {"Method": "Vanilla (LogReg + BERT)",
         "Accuracy": f"{_v_acc:.4f}", "F1 (macro)": f"{_v_f1:.4f}",
         "Avg Confidence": f"{_v_conf:.4f}",
         "Clarif. Rate": "0.0%", "Avg Clarif. Turns": "0.00",
         "Samples": len(_dfv)},
        {"Method": "DS + Optimal Thresholds",
         "Accuracy": f"{_d_acc:.4f}", "F1 (macro)": f"{_d_f1:.4f}",
         "Avg Confidence": f"{_d_conf:.4f}",
         "Clarif. Rate": f"{_clarif_rate:.1f}%" if not np.isnan(_clarif_rate) else "N/A",
         "Avg Clarif. Turns": f"{_avg_clarif:.2f}" if not np.isnan(_avg_clarif) else "N/A",
         "Samples": len(_dfd)},
    ])
    print(_tbl.to_string(index=False))
    print(f"\n  📈 Accuracy delta : {_acc_delta*100:+.2f}%   |   F1 delta: {_f1_delta*100:+.2f}%")
    if _mcn:
        _sig = "✅ YES" if _mcn.get('significant_at_0.05') else "❌ NO"
        print(f"  McNemar p={_mcn.get('mcnemar_pvalue', float('nan')):.4f}  Significant: {_sig}  "
              f"Net: {_mcn.get('net_improvement', '?'):+}  →  {_mcn.get('conclusion', 'N/A')}")

    # ── save per-dataset files ────────────────────────────────────────────────
    _tbl.to_csv(_cmp_dir / "comparison_table.csv", index=False)
    _out = {
        "dataset": _ds,
        "vanilla": {"accuracy": _v_acc, "f1_macro": _v_f1, "avg_confidence": _v_conf, "samples": len(_dfv)},
        "ds":      {"accuracy": _d_acc, "f1_macro": _d_f1, "avg_confidence": _d_conf,
                    "clarification_rate_pct": _clarif_rate, "avg_clarif_turns": _avg_clarif, "samples": len(_dfd)},
        "delta":   {"accuracy": _acc_delta, "f1_macro": _f1_delta},
        "mcnemar": _mcn,
    }
    with open(_cmp_dir / "all_results.json", "w") as _fh:
        json.dump(_out, _fh, indent=2, default=float)
    _all_outputs[_ds] = _out

    _per_dataset_rows.append({
        "Dataset":      _ds,
        "Vanilla Acc":  f"{_v_acc:.4f}",
        "DS Acc":       f"{_d_acc:.4f}",
        "Δ Acc":        f"{_acc_delta*100:+.2f}%",
        "DS F1":        f"{_d_f1:.4f}",
        "Δ F1":         f"{_f1_delta*100:+.2f}%",
        "Clarif. Rate": f"{_clarif_rate:.1f}%" if not np.isnan(_clarif_rate) else "N/A",
        "p-value":      f"{_mcn.get('mcnemar_pvalue', float('nan')):.4f}",
        "Sig.":         "✅" if _mcn.get('significant_at_0.05') else "❌",
        "Conclusion":   _mcn.get('conclusion', 'N/A'),
    })

# ── final cross-dataset summary ───────────────────────────────────────────────
if _per_dataset_rows:
    print(f"\n{'='*70}")
    print("CROSS-DATASET SUMMARY")
    print(f"{'='*70}")
    print(pd.DataFrame(_per_dataset_rows).to_string(index=False))
    print(f"{'='*70}")
    print("\n✅ PHASE 5 COMPLETE!  Results saved to results/{{dataset}}/workflow_demo/comparison/")


PHASE 5: COMPREHENSIVE COMPARISON  (Banking77 + Clinc150)

──────────────────────────────────────────────────────────────────────
  BANKING77
──────────────────────────────────────────────────────────────────────
                 Method Accuracy F1 (macro) Avg Confidence Clarif. Rate Avg Clarif. Turns  Samples
Vanilla (LogReg + BERT)   0.8766     0.8728         0.3980         0.0%              0.00     3080
DS + Optimal Thresholds   0.8951     0.8914         0.4413        13.7%              0.29     3080

  📈 Accuracy delta : +1.85%   |   F1 delta: +1.86%
  McNemar p=0.0000  Significant: ✅ YES  Net: +57  →  DS better

──────────────────────────────────────────────────────────────────────
  CLINC150
──────────────────────────────────────────────────────────────────────
                 Method Accuracy F1 (macro) Avg Confidence Clarif. Rate Avg Clarif. Turns  Samples
Vanilla (LogReg + BERT)   0.9464     0.9464         0.3958         0.0%              0.00     4500
DS + Optimal Thresholds

## Runtime Session Export And Post-Study Analysis
Use these cells after collecting real participant sessions in the private repo. This workflow exports session JSON files to Excel and runs conflict/ACC analyses from the exported data.

In [ ]:
from pathlib import Path
import subprocess
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPORT_SCRIPT = PROJECT_ROOT / "scripts/analysis/export_sessions_to_excel.py"
EXPORT_OUTPUT_DIR = PROJECT_ROOT / "outputs/session_exports"

print(f"Project root: {PROJECT_ROOT}")
print(f"Export script: {EXPORT_SCRIPT}")
print(f"Export output dir: {EXPORT_OUTPUT_DIR}")

cmd = [
    "python",
    str(EXPORT_SCRIPT),
    "--output-dir", str(EXPORT_OUTPUT_DIR),
]

print("Running export_sessions_to_excel.py ...")
result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Export failed with exit code {result.returncode}")

print("Export complete.")
print(f"Expected workbook: {EXPORT_OUTPUT_DIR / 'sessions_export.xlsx'}")

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFLICT_SCRIPT = PROJECT_ROOT / "scripts/analysis/analyze_conflict_zadeh.py"
ACC_SCRIPT = PROJECT_ROOT / "scripts/analysis/analyze_acc_curves.py"

SESSIONS_DIR = PROJECT_ROOT / "outputs/session_exports/sessions/sessions/clinc150"
CONFLICT_OUT = PROJECT_ROOT / "outputs/zadeh_conflict_sessions_live"
RUN_ACC_CURVES = True

print(f"Sessions source: {SESSIONS_DIR}")
print(f"Conflict script: {CONFLICT_SCRIPT}")
print(f"Conflict output: {CONFLICT_OUT}")

conflict_cmd = [
    "python",
    str(CONFLICT_SCRIPT),
    "--sessions-dir", str(SESSIONS_DIR),
    "--dataset", "clinc150",
    "--output-dir", str(CONFLICT_OUT),
    "--rules", "dempster,no_norm,yager",
]

print("Running analyze_conflict_zadeh.py ...")
res_conflict = subprocess.run(conflict_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(res_conflict.stdout)
if res_conflict.returncode != 0:
    print(res_conflict.stderr)
    raise RuntimeError(f"Conflict analysis failed with exit code {res_conflict.returncode}")

print("Conflict analysis complete.")
print(f"Overall: {CONFLICT_OUT / 'ablation_overall.csv'}")
print(f"Cluster: {CONFLICT_OUT / 'ablation_cluster.csv'}")
print(f"Intent: {CONFLICT_OUT / 'ablation_intent.csv'}")
print(f"Session: {CONFLICT_OUT / 'ablation_session.csv'}")

if RUN_ACC_CURVES:
    acc_xlsx = PROJECT_ROOT / "outputs/session_exports/sessions_export.xlsx"
    acc_csv = PROJECT_ROOT / "outputs/session_exports/query_level_from_sessions_export.csv"
    acc_out = PROJECT_ROOT / "outputs/acc_curves_live"

    print("\nPreparing ACC input from sessions_export.xlsx ...")
    if not acc_xlsx.exists():
        print(f"Skipping ACC curves because workbook is missing: {acc_xlsx}")
    else:
        query_df = pd.read_excel(acc_xlsx, sheet_name="query_level")
        query_df.to_csv(acc_csv, index=False)
        print(f"Saved ACC input CSV: {acc_csv}")

        print("Running analyze_acc_curves.py ...")
        acc_cmd = [
            "python",
            str(ACC_SCRIPT),
            "--results-file", str(acc_csv),
            "--output-dir", str(acc_out),
            "--target-coverage", "0.9",
            "--target-accuracy", "0.85",
        ]
        res_acc = subprocess.run(acc_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
        print(res_acc.stdout)
        if res_acc.returncode != 0:
            print(res_acc.stderr)
            raise RuntimeError(f"ACC analysis failed with exit code {res_acc.returncode}")
        print(f"ACC output: {acc_out}")

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFLICT_SCRIPT = PROJECT_ROOT / "scripts/analysis/analyze_conflict_zadeh.py"
ACC_SCRIPT = PROJECT_ROOT / "scripts/analysis/analyze_acc_curves.py"

SESSIONS_DIR = PROJECT_ROOT / "outputs/session_exports/sessions/sessions/b77"
CONFLICT_OUT = PROJECT_ROOT / "outputs/zadeh_conflict_sessions_live_b77"
RUN_ACC_CURVES = True

print(f"Sessions source: {SESSIONS_DIR}")
print(f"Conflict script: {CONFLICT_SCRIPT}")
print(f"Conflict output: {CONFLICT_OUT}")

conflict_cmd = [
    "python",
    str(CONFLICT_SCRIPT),
    "--sessions-dir", str(SESSIONS_DIR),
    "--dataset", "banking77",
    "--output-dir", str(CONFLICT_OUT),
    "--rules", "dempster,no_norm,yager",
]

print("Running analyze_conflict_zadeh.py (B77) ...")
res_conflict = subprocess.run(conflict_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(res_conflict.stdout)
if res_conflict.returncode != 0:
    print(res_conflict.stderr)
    raise RuntimeError(f"Conflict analysis failed with exit code {res_conflict.returncode}")

print("Conflict analysis complete.")
print(f"Overall: {CONFLICT_OUT / 'ablation_overall.csv'}")
print(f"Cluster: {CONFLICT_OUT / 'ablation_cluster.csv'}")
print(f"Intent: {CONFLICT_OUT / 'ablation_intent.csv'}")
print(f"Session: {CONFLICT_OUT / 'ablation_session.csv'}")

if RUN_ACC_CURVES:
    acc_xlsx = PROJECT_ROOT / "outputs/session_exports/sessions_export.xlsx"
    acc_csv = PROJECT_ROOT / "outputs/session_exports/query_level_from_sessions_export_b77.csv"
    acc_out = PROJECT_ROOT / "outputs/acc_curves_live_b77"

    print("\nPreparing ACC input from sessions_export.xlsx ...")
    if not acc_xlsx.exists():
        print(f"Skipping ACC curves because workbook is missing: {acc_xlsx}")
    else:
        query_df = pd.read_excel(acc_xlsx, sheet_name="query_level")
        query_df = query_df[query_df.get("dataset", "") == "banking77"].copy()
        query_df.to_csv(acc_csv, index=False)
        print(f"Saved ACC input CSV: {acc_csv}")

        print("Running analyze_acc_curves.py (B77) ...")
        acc_cmd = [
            "python",
            str(ACC_SCRIPT),
            "--results-file", str(acc_csv),
            "--output-dir", str(acc_out),
            "--target-coverage", "0.9",
            "--target-accuracy", "0.85",
        ]
        res_acc = subprocess.run(acc_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
        print(res_acc.stdout)
        if res_acc.returncode != 0:
            print(res_acc.stderr)
            raise RuntimeError(f"ACC analysis failed with exit code {res_acc.returncode}")
        print(f"ACC output: {acc_out}")